In [1]:
import sys

from pathlib import Path


PROJECT_ROOT = Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))


from agentic_ai.proton import (
    open_proton_mail,
    close_proton_mail,
    ensure_logged_in,
    is_trash_page,
    open_trash,
    return_to_trash,
    open_trash_conversation,
    star_conversation,
)

from agentic_ai.agent import (
    create_email_triage_agent,
)

from agentic_ai.models import (
    EmailImportance,
)


In [2]:

agents_file = PROJECT_ROOT / "src" / "agentic_ai" / "agents.py"

print("Path:", agents_file)
print("Exists:", agents_file.exists())

Path: /Users/theo/Documents/Projects/programingProjects/agentic_AI_project/agentic-ai/src/agentic_ai/agents.py
Exists: False


### Create the email agent

In [3]:
email_agent = create_email_triage_agent()

In [4]:
playwright, context, page = (
    await open_proton_mail(
        project_root=PROJECT_ROOT,
    )
)

await ensure_logged_in(page)


Proton Mail login is required.
Log in manually in the browser window.
Login detected.


In [5]:
await open_trash(page)

Opened Trash.


In [6]:
print(
    "Trash:",
    await is_trash_page(page),
)

Trash: True


In [7]:
rows = page.locator(
    "div.item-container--row"
)

first_row = rows.nth(0)

star_button = first_row.locator(
    "button.starbutton"
)

print("Star buttons:", await star_button.count())

if await star_button.count() == 1:
    print(
        await star_button.evaluate(
            """
            el => ({
                text: (el.innerText || "").trim(),
                ariaLabel: el.getAttribute("aria-label"),
                title: el.getAttribute("title"),
                className: el.className
            })
            """
        )
    )

Star buttons: 1
{'text': 'Star conversation', 'ariaLabel': None, 'title': None, 'className': 'starbutton relative item-star flex stop-propagation'}


In [8]:
await star_conversation(
    page,
    index=0,
)

Conversation 0 is now starred.


In [9]:
await star_conversation(
    page,
    index=999999,
)

IndexError: Conversation index 999999 is invalid. Valid indexes: 0-49.

In [10]:
email = await open_trash_conversation(
    page,
    index=0,
)

In [11]:
await star_conversation(
    page,
    index=0,
)

RuntimeError: The current Proton page is not Trash.

In [12]:
result = await email_agent.run(
    f"""
    Classify the importance of this email.

    Use these rules:

    "ignore":
    The email does not appear to require attention.

    "review":
    The email is potentially useful or relevant,
    but there is no clear urgent action.

    "urgent":
    The email requires timely attention,
    contains an important deadline,
    or could cause a significant consequence
    if ignored.

    Set action_required to true only when the email
    clearly requires the recipient to do something.

    Give one concise factual reason for the classification.

    Do not invent information that is not present.

    Email:

    {email.raw_text}
    """
)

                 pydantic-ai v2.50.0 • Python 3.13.15

      / \        agent: email_agent • model: ollama:qwen3:4b-instruct • output: EmailImportance
     /   \         tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


In [13]:
print(type(result.output))
print("Priority:", result.output.priority)
print( "Action required:", result.output.action_required,)
print("Reason:", result.output.reason)

<class 'agentic_ai.models.EmailImportance'>
Priority: review
Action required: False
Reason: The email is a notification from a mailing list about job postings, which is potentially useful but does not require immediate action or contain a deadline.


In [14]:
import sys

print(sys.executable)
print(sys.version)

/opt/anaconda3/envs/agentic-ai/bin/python
3.13.15 | packaged by Anaconda, Inc. | (main, Aug 14 2026, 17:19:34) [Clang 20.1.8 ]


````
conda activate agentic-ai   
```

```
conda deactivate
```